# Clustering — K-Means y Jerárquico

_Distancias, elección de K, dendrogramas y un caso real de segmentación de clientes_

**Módulo 2 — Aprendizaje No Supervisado** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

![Aprendizaje No Supervisado](assets/header.png)

## 1. ¿Qué es clustering?

Dado un conjunto de observaciones $\{x_1, \dots, x_n\}$ sin etiquetas, queremos partirlas en $K$ grupos tales que:

- Los puntos **dentro** de un grupo se parezcan entre sí (alta cohesión interna).
- Los puntos de **distintos** grupos sean diferentes (alta separación entre clusters).

Formalmente buscamos una partición $C_1, \dots, C_K$ que minimice una medida de **disimilitud intra-cluster**:

$$
\min_{C_1,\dots,C_K} \; \sum_{k=1}^{K} \sum_{x_i \in C_k} d(x_i, \mu_k)
$$

donde $\mu_k$ es el centro del cluster $k$ y $d(\cdot,\cdot)$ es una métrica de distancia.

## 2. Métricas de distancia

La métrica condiciona qué se considera "parecido". Las más usadas:

**Distancia euclidiana** (la más común, geometría natural):
$$
d_{\text{euc}}(x, y) = \sqrt{\sum_{j=1}^{p} (x_j - y_j)^2}
$$

**Distancia de Manhattan** (más robusta a outliers en ejes individuales):
$$
d_{\text{man}}(x, y) = \sum_{j=1}^{p} |x_j - y_j|
$$

**Distancia de Minkowski** (familia general, $p=2$ → euclidiana, $p=1$ → Manhattan):
$$
d_{\text{mink}}(x, y) = \Big(\sum_{j=1}^{p} |x_j - y_j|^q\Big)^{1/q}
$$

**Coseno** (mide el ángulo, ignora la magnitud — clave para texto y embeddings):
$$
d_{\cos}(x, y) = 1 - \frac{x \cdot y}{\|x\| \, \|y\|}
$$

**Mahalanobis** (toma en cuenta la covarianza entre variables):
$$
d_{\text{mah}}(x, y) = \sqrt{(x - y)^\top \Sigma^{-1} (x - y)}
$$

> ⚠️ **K-Means** está atado a la distancia euclidiana (los centroides son medias). Para usar otras distancias se prefieren **K-Medoids** o el **clustering jerárquico**, que sí acepta cualquier métrica.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

# Visualizamos cómo cambia el "vecino más cercano" según la métrica
rng = np.random.default_rng(0)
puntos = rng.uniform(0, 10, size=(40, 2))
ref = np.array([[5, 5]])

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axes, ['euclidean', 'cityblock', 'cosine']):
    d = cdist(ref, puntos, metric=metric).ravel()
    sc = ax.scatter(puntos[:, 0], puntos[:, 1], c=d, cmap='viridis', s=60)
    ax.scatter(*ref[0], marker='*', s=300, c='red', edgecolor='black')
    ax.set_title(f'Distancia: {metric}')
    plt.colorbar(sc, ax=ax)
plt.tight_layout(); plt.show()

## 3. K-Means

### 3.1 Idea

Encuentra $K$ centroides $\mu_1, \dots, \mu_K$ y asigna cada punto al centroide más cercano. Minimiza la **inercia** (suma de cuadrados intra-cluster):

$$
J = \sum_{k=1}^{K} \sum_{x_i \in C_k} \| x_i - \mu_k \|^2
$$

### 3.2 Algoritmo (Lloyd)

1. Inicializar $K$ centroides (idealmente con `k-means++` para una mejor partida).
2. **Asignación**: cada $x_i$ va al cluster del centroide más cercano.
3. **Actualización**: $\mu_k$ se recalcula como la media de los puntos asignados a $C_k$.
4. Repetir 2–3 hasta que los centroides dejen de moverse (o se alcancen las iteraciones máximas).

### 3.3 Limitaciones a tener en cuenta

- Necesita que **fijes K** de antemano.
- Asume clusters **esféricos** y de tamaño parecido (por usar distancia euclidiana).
- Sensible a la **inicialización** (`n_init` repite varias veces y se queda con el mejor).
- Sensible a la **escala** y a los **outliers** (mejor con `StandardScaler`).

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

X, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.9, random_state=0)
km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X[:, 0], X[:, 1], c=km.labels_, cmap='tab10', s=25, alpha=0.8)
ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
           marker='X', c='black', s=200, label='centroides')
ax.set_title('K-Means con K=4 sobre datos sintéticos')
ax.legend(); plt.show()

## 4. ¿Cómo elegir el K óptimo?

No hay una respuesta universal. Las tres heurísticas más usadas:

### 4.1 Método del codo (elbow)

Graficamos la **inercia** $J(K)$ contra $K$. Como $J$ siempre baja al aumentar $K$, buscamos el "codo" — el punto donde la mejora marginal se aplana.

### 4.2 Coeficiente de silueta

Para cada punto $i$:

$$
s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}
$$

donde $a(i)$ es la distancia media al **mismo** cluster y $b(i)$ es la distancia media al cluster vecino más cercano. La silueta promedio se mueve en $[-1, 1]$: cuanto más cerca de 1, mejor separados están los clusters. Elegimos el $K$ que **maximiza** el promedio.

### 4.3 Gap statistic (Tibshirani)

Compara la inercia observada con la esperada bajo una distribución nula (datos uniformes). Elige el $K$ donde la "brecha" (gap) es mayor. Más robusta que el codo pero más cara de computar.

In [ ]:
from sklearn.metrics import silhouette_score

Ks = range(2, 11)
inercias, siluetas = [], []
for k in Ks:
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    inercias.append(m.inertia_)
    siluetas.append(silhouette_score(X, m.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(Ks, inercias, marker='o')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inercia (J)')
axes[0].set_title('Método del codo')
axes[0].grid(True)

axes[1].plot(Ks, siluetas, marker='o', color='darkgreen')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette promedio')
axes[1].set_title('Coeficiente de silueta')
axes[1].grid(True)
plt.tight_layout(); plt.show()

print(f'K con mayor silueta: K={Ks[int(np.argmax(siluetas))]}')

## 5. Clustering jerárquico

A diferencia de K-Means, **no requiere fijar K** de antemano: produce un árbol completo (**dendrograma**) que muestra cómo se van fusionando los puntos. Hay dos estrategias:

- **Aglomerativo (bottom-up)** — el más usado. Cada punto empieza siendo su propio cluster y se van fusionando los pares más parecidos.
- **Divisivo (top-down)** — todos los puntos arrancan en un solo cluster y se van separando.

### 5.1 Linkage — cómo medir la distancia entre clusters

La métrica de distancia entre puntos es solo la mitad: hay que decidir cómo medir la distancia entre **grupos**.

| Linkage | Distancia entre clusters $A$ y $B$ |
|---|---|
| **Single** (mín.) | $\min_{a \in A, b \in B} d(a, b)$ — tiende a clusters alargados ("efecto cadena") |
| **Complete** (máx.) | $\max_{a \in A, b \in B} d(a, b)$ — clusters compactos |
| **Average** | promedio de todas las distancias entre pares |
| **Ward** | minimiza el incremento de la suma de cuadrados intra-cluster (similar a K-Means) |

`Ward` es el default razonable cuando los datos están escalados y queremos clusters de tamaño parecido.

### 5.2 Dendrograma — cómo elegir K

El dendrograma se "corta" a una altura $h$: todas las ramas que la línea cruza son clusters distintos. Una heurística clásica es cortar en el **mayor salto vertical** sin cruce horizontal.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# Submuestra para que el dendrograma sea legible
idx = np.random.default_rng(0).choice(len(X), size=80, replace=False)
Z = linkage(X[idx], method='ward')

fig, ax = plt.subplots(figsize=(11, 5))
dendrogram(Z, ax=ax, color_threshold=12)
ax.axhline(12, color='red', ls='--', label='corte sugerido')
ax.set_title('Dendrograma — clustering jerárquico aglomerativo (linkage=Ward)')
ax.set_xlabel('observaciones'); ax.set_ylabel('distancia de fusión')
ax.legend(); plt.show()

labels_h = fcluster(Z, t=4, criterion='maxclust')
print('Clusters obtenidos al cortar en K=4:', np.unique(labels_h))

## 6. Caso práctico — Telco Customer Churn

**Dataset:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn
**Archivo:** `WA_Fn-UseC_-Telco-Customer-Churn.csv` (7.043 × 21).

En este módulo **no estamos prediciendo** `Churn`. Lo que queremos es descubrir **segmentos naturales de clientes** a partir de su comportamiento de uso y facturación. Para mantener el ejemplo interpretable usamos solo variables numéricas:

- `tenure` — meses como cliente.
- `MonthlyCharges` — cuota mensual (USD).
- `TotalCharges` — facturación acumulada (USD).

Al final usaremos `Churn` solo como **validación externa** para ver si los segmentos descubiertos tienen sentido.

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
if not DATA.exists():
    raise FileNotFoundError(
        f'No se encontró {DATA}. Descarga el dataset desde '
        'https://www.kaggle.com/datasets/blastchar/telco-customer-churn '
        'y colócalo en data/ (ver README.md).'
    )

df = pd.read_csv(DATA)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges']).reset_index(drop=True)
print('Shape:', df.shape)
df.head()

In [ ]:
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid')

features = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_raw = df[features].copy()

scaler = StandardScaler().fit(X_raw)
X_s = scaler.transform(X_raw)
print('Variables escaladas: media≈0, std≈1')
pd.DataFrame(X_s, columns=features).describe().round(2).loc[['mean', 'std']]

In [ ]:
# --- Elegir K con codo + silueta ---
Ks = range(2, 9)
inercias, siluetas = [], []
for k in Ks:
    m = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_s)
    inercias.append(m.inertia_)
    siluetas.append(silhouette_score(X_s, m.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(Ks, inercias, marker='o')
axes[0].set_title('Codo'); axes[0].set_xlabel('K'); axes[0].set_ylabel('Inercia')
axes[0].grid(True)

axes[1].plot(Ks, siluetas, marker='o', color='darkgreen')
axes[1].set_title('Silueta'); axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette')
axes[1].grid(True)
plt.tight_layout(); plt.show()

K = Ks[int(np.argmax(siluetas))]
print(f'K elegido (max silueta): {K}')

In [ ]:
# --- K-Means final ---
km = KMeans(n_clusters=K, n_init=20, random_state=42).fit(X_s)
df['cluster_km'] = km.labels_

# Centroides en escala original
centros = pd.DataFrame(
    scaler.inverse_transform(km.cluster_centers_),
    columns=features,
).round(1)
centros.index.name = 'cluster'
centros['n_clientes'] = df['cluster_km'].value_counts().sort_index().values
centros

In [ ]:
# --- Clustering jerárquico (Ward) sobre una submuestra para comparar ---
sub = df.sample(n=1000, random_state=42)
X_sub = scaler.transform(sub[features])

agg = AgglomerativeClustering(n_clusters=K, linkage='ward').fit(X_sub)
sub['cluster_h'] = agg.labels_

# Comparamos asignaciones
sub['cluster_km'] = km.predict(X_sub)
ct = pd.crosstab(sub['cluster_km'], sub['cluster_h'],
                 rownames=['K-Means'], colnames=['Jerárquico'])
print('Concordancia entre los dos métodos sobre la submuestra:')
ct

In [ ]:
# --- Visualización: pares de variables coloreados por cluster ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pares = [('tenure', 'MonthlyCharges'),
         ('tenure', 'TotalCharges'),
         ('MonthlyCharges', 'TotalCharges')]
for ax, (a, b) in zip(axes, pares):
    sns.scatterplot(x=a, y=b, hue='cluster_km', data=df,
                    palette='tab10', s=15, alpha=0.6, ax=ax, legend=False)
    ax.set_title(f'{a} vs {b}')
plt.tight_layout(); plt.show()

### Interpretación de los segmentos

A partir de los centroides en escala original, podemos darle un nombre comercial a cada cluster (los nombres dependen del K que haya salido — adapta la lectura a tus centroides):

- **Clientes nuevos / bajo gasto**: `tenure` corto, `TotalCharges` bajo.
- **Clientes leales / alto valor**: `tenure` alto, `TotalCharges` alto.
- **Clientes premium recientes**: `MonthlyCharges` alto pero `tenure` corto — son los más riesgosos: pagan caro y aún no se "atan" a la compañía.

Esto es lo que vuelve útil el clustering: **no es predecir** sino **describir** la base de clientes para diseñar acciones diferenciadas.

In [ ]:
# --- Validación externa con el target real (que NO usamos para entrenar) ---
df['Churn_bin'] = (df['Churn'] == 'Yes').astype(int)
churn_por_cluster = df.groupby('cluster_km').agg(
    n_clientes=('Churn_bin', 'size'),
    tasa_churn=('Churn_bin', 'mean'),
).round(3)
churn_por_cluster

Si la **tasa de churn varía mucho entre clusters**, los segmentos detectados están capturando algo real del negocio aunque nunca usamos `Churn` para entrenar. Esto es la mejor validación posible para un problema no supervisado: el patrón se sostiene contra una etiqueta independiente.

## 7. Referencias

- ISLR cap. 12.4 — *Clustering Methods*. https://www.statlearning.com/
- Lloyd, S. (1982). *Least squares quantization in PCM* (algoritmo K-Means).
- Ward, J. H. (1963). *Hierarchical grouping to optimize an objective function*.
- Rousseeuw, P. (1987). *Silhouettes: A graphical aid to the interpretation and validation of cluster analysis*.
- Tibshirani, R., Walther, G. & Hastie, T. (2001). *Estimating the number of clusters in a data set via the gap statistic*.
- scikit-learn: https://scikit-learn.org/stable/modules/clustering.html
- Dataset: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

## Predicción sobre datos nuevos — uso del modelo en producción

Aunque "predecir" en clustering significa cosas distintas según el algoritmo, sí podemos persistir un modelo y asignar nuevos clientes a los segmentos descubiertos:

1. **Reentrenar con todos los datos** (ya validamos K, ahora aprovechamos toda la información).
2. **Persistir el `scaler` junto al modelo** — sin la misma escala, las distancias no significan lo mismo.
3. **Asignar** cada nuevo cliente al cluster del centroide más cercano con `kmeans.predict(...)`.

> Limitación: K-Means tiene `predict`, pero **el clustering jerárquico no** — habría que reajustarlo con el nuevo punto incluido o entrenar un clasificador supervisado sobre los labels.

In [ ]:
import joblib

# Reentrenamos con TODOS los datos disponibles
scaler_final = StandardScaler().fit(df[features])
modelo_final = KMeans(n_clusters=K, n_init=20, random_state=42).fit(
    scaler_final.transform(df[features])
)

bundle = {
    'modelo':   modelo_final,
    'scaler':   scaler_final,
    'features': features,
    'K':        K,
}
joblib.dump(bundle, 'modelo_telco_kmeans.pkl')
print('Bundle guardado: modelo_telco_kmeans.pkl')

### Inferencia individual — un cliente nuevo

In [ ]:
nuevo_cliente = pd.DataFrame([{
    'tenure':         3,
    'MonthlyCharges': 95.0,
    'TotalCharges':   285.0,
}])

cluster_id = modelo_final.predict(scaler_final.transform(nuevo_cliente))[0]
print(f'El cliente cae en el cluster: {cluster_id}')
print('Centroide de ese cluster (escala original):')
print(centros.loc[cluster_id])

**Lecciones para producción:**
- El `K` óptimo depende de qué decisiones de negocio quieres soportar — una segmentación con K=4 puede ser estadísticamente buena pero inútil si marketing solo puede ejecutar 2 campañas.
- Refrescar los clusters cada cierto tiempo: el comportamiento de los clientes cambia (data drift). Un cluster que existía el año pasado puede haberse difuminado.
- Documentar **qué significa cada cluster** en lenguaje de negocio, no como "cluster 0, cluster 1".